In [ ]:
import pandas as pd 
import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel('data/BAT1.xlsx', sheet_name= 'british_airways_schedule_summer')
df.sample(5)

,FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,AIRCRAFT_TYPE,FIRST_CLASS_SEATS,BUSINESS_CLASS_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX
6201,2025-07-12,20:36:00,Evening,BA,BA3365,LHR,MAD,Spain,Europe,SHORT,A320,0,20,160,0,2,13
6991,2025-08-14,14:39:00,Afternoon,BA,BA8887,LHR,JFK,USA,North America,LONG,B777,8,49,178,0,13,45
5407,2025-10-06,07:20:00,Morning,BA,BA6527,LHR,FRA,Germany,Europe,SHORT,A320,0,5,175,1,14,48
4309,2025-05-22,08:10:00,Morning,BA,BA3030,LHR,MAD,Spain,Europe,SHORT,A320,0,6,174,2,5,23
1757,2025-09-16,10:03:00,Morning,BA,BA8174,LHR,MAD,Spain,Europe,SHORT,A320,0,19,161,1,13,45


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   FLIGHT_DATE           10000 non-null  datetime64[us]
 1   FLIGHT_TIME           10000 non-null  object        
 2   TIME_OF_DAY           10000 non-null  str           
 3   AIRLINE_CD            10000 non-null  str           
 4   FLIGHT_NO             10000 non-null  str           
 5   DEPARTURE_STATION_CD  10000 non-null  str           
 6   ARRIVAL_STATION_CD    10000 non-null  str           
 7   ARRIVAL_COUNTRY       10000 non-null  str           
 8   ARRIVAL_REGION        10000 non-null  str           
 9   HAUL                  10000 non-null  str           
 10  AIRCRAFT_TYPE         10000 non-null  str           
 11  FIRST_CLASS_SEATS     10000 non-null  int64         
 12  BUSINESS_CLASS_SEATS  10000 non-null  int64         
 13  ECONOMY_SEATS         10000 

In [4]:
df['TIME_OF_DAY'].value_counts()

TIME_OF_DAY
Morning      3530
Evening      2973
Afternoon    2305
Lunchtime    1192
Name: count, dtype: int64

In [5]:
df_country_seats = df.groupby('ARRIVAL_COUNTRY')['ECONOMY_SEATS'].sum().sort_values(ascending=False)

display(df_country_seats)

ARRIVAL_COUNTRY
USA            639772
Germany        239144
Spain          223998
UAE            165802
Japan          162711
Austria        115808
Turkey         110515
Netherlands    108774
France         108739
Switzerland    108596
Name: ECONOMY_SEATS, dtype: int64

In [6]:
df_country_region = df.groupby('ARRIVAL_COUNTRY')['ARRIVAL_REGION'].describe().sort_values(by='count', ascending=False)

display(df_country_region)

,count,unique,top,freq
ARRIVAL_COUNTRY,,,,
USA,2658,1,North America,2658
Germany,1405,1,Europe,1405
Spain,1318,1,Europe,1318
UAE,688,1,Middle East,688
Austria,682,1,Europe,682
Japan,679,1,Asia,679
Turkey,650,1,Europe,650
France,641,1,Europe,641
Netherlands,641,1,Europe,641


In [7]:
df_haul_time = (
    df.groupby(['HAUL', 'TIME_OF_DAY'])
      .size()
      .reset_index(name='NUM_FLIGHTS')
)

display(df_haul_time)

,HAUL,TIME_OF_DAY,NUM_FLIGHTS
0,LONG,Afternoon,939
1,LONG,Evening,1190
2,LONG,Lunchtime,435
3,LONG,Morning,1461
4,SHORT,Afternoon,1366
5,SHORT,Evening,1783
6,SHORT,Lunchtime,757
7,SHORT,Morning,2069


In [8]:
df["CATEGORY"] = df["HAUL"] + " " + df["TIME_OF_DAY"]

In [9]:
df['CATEGORY'].value_counts()

CATEGORY
SHORT Morning      2069
SHORT Evening      1783
LONG Morning       1461
SHORT Afternoon    1366
LONG Evening       1190
LONG Afternoon      939
SHORT Lunchtime     757
LONG Lunchtime      435
Name: count, dtype: int64

In [10]:
lookup = {
    "LONG Morning": {
        "Tier1": 0.015,
        "Tier2": 0.040,
        "Tier3": 0.180
    },
    "LONG Afternoon": {
        "Tier1": 0.012,
        "Tier2": 0.038,
        "Tier3": 0.160
    },
    "LONG Lunchtime": {
        "Tier1": 0.010,
        "Tier2": 0.035,
        "Tier3": 0.150
    },
    "LONG Evening": {
        "Tier1": 0.008,
        "Tier2": 0.030,
        "Tier3": 0.130
    },

    "SHORT Morning": {
        "Tier1": 0.003,
        "Tier2": 0.020,
        "Tier3": 0.080
    },
    "SHORT Afternoon": {
        "Tier1": 0.0025,
        "Tier2": 0.018,
        "Tier3": 0.070
    },
    "SHORT Lunchtime": {
        "Tier1": 0.002,
        "Tier2": 0.015,
        "Tier3": 0.060
    },
    "SHORT Evening": {
        "Tier1": 0.001,
        "Tier2": 0.010,
        "Tier3": 0.040
    }
}

In [24]:
df["TIER 1 %"] = df["CATEGORY"].map(
    lambda x: round(lookup[x]["Tier1"] * 100, 3)
)

df["TIER 2 %"] = df["CATEGORY"].map(
    lambda x: round(lookup[x]["Tier2"] * 100, 3)
)

df["TIER 3 %"] = df["CATEGORY"].map(
    lambda x: round(lookup[x]["Tier3"] * 100, 3)
)

In [25]:
df_lookup = df[['FLIGHT_DATE', 'FLIGHT_TIME', 'ARRIVAL_COUNTRY', 'CATEGORY', 'TIER 1 %', 'TIER 2 %', 'TIER 3 %']]

In [26]:
df_lookup.sample(5)

,FLIGHT_DATE,FLIGHT_TIME,ARRIVAL_COUNTRY,CATEGORY,TIER 1 %,TIER 2 %,TIER 3 %
5557,2025-06-25,18:04:00,Spain,SHORT Evening,0.10,1.0,4.0
5016,2025-09-05,20:51:00,USA,LONG Evening,0.80,3.0,13.0
5088,2025-09-10,17:11:00,Austria,SHORT Afternoon,0.25,1.8,7.0
2036,2025-04-23,10:32:00,USA,LONG Morning,1.50,4.0,18.0
8798,2025-05-07,20:50:00,Spain,SHORT Evening,0.10,1.0,4.0


In [27]:
df_lookup.to_excel('data/df_lookup.xlsx')